# CNN Car Image Classifier — 7 Classes

**PPR501 — Machine Learning for Engineers (Applied Machine Learning)**

Objective: classify a car photograph into one of seven brand groups.

> 📌 **About the word "CNN" in the title:** the model used here is **MobileNetV2** —
> which **is** a CNN (Convolutional Neural Network), just a lightweight optimised
> variant that is common on phones. This section of the project is the
> **car-image recognition** part built with a CNN.

| # | Class |
|---|-------|
| 0 | Audi |
| 1 | Hyundai Creta |
| 2 | Mahindra Scorpio |
| 3 | Rolls Royce |
| 4 | Maruti Suzuki Swift |
| 5 | Tata Safari |
| 6 | Toyota Innova |

## How to run

This notebook is **fully self-contained** — it does not import anything from the
rest of the project. Every piece of logic lives directly inside the notebook.

**Requirements:** TensorFlow, Keras, NumPy, Matplotlib, Scikit-learn, Gradio, Pillow.

```bash
pip install tensorflow numpy matplotlib scikit-learn gradio pillow
```

**The dataset** must be laid out as `sources/Dataset/Train/<Class>/*.jpg` and
`sources/Dataset/Test/<Class>/*.jpg`. The path is located automatically in the
cell below.

> ⏱️ Training time: roughly **11 minutes** on a laptop CPU, 20 epochs.

In [ ]:
# ============================================================
# CELL 1 - Imports
# ============================================================
import os
import json
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau,
)
from sklearn.metrics import classification_report, confusion_matrix, recall_score
from PIL import Image

print(f"TensorFlow {tf.__version__}")

## 1. Configuration

All hyper-parameters are collected in one place so they are easy to adjust.

In [ ]:
# ============================================================
# CELL 2 - Configuration
# ============================================================
SEED = 42
IMAGE_SIZE = 224          # MobileNetV2 default input size
BATCH_SIZE = 32
NUM_CLASSES = 7
VALIDATION_SPLIT = 0.2   # 20% of Train is held out for validation

EPOCHS_FROZEN = 5        # Stage 1: frozen backbone
EPOCHS_FINETUNE = 15     # Stage 2: unfrozen backbone
LR_STAGE1 = 1e-3
LR_STAGE2 = 1e-4
PATIENCE = 6

CLASS_NAMES = [
    "Audi",
    "Hyundai Creta",
    "Mahindra Scorpio",
    "Rolls Royce",
    "Maruti Suzuki Swift",
    "Tata Safari",
    "Toyota Innova",
]

tf.keras.utils.set_random_seed(SEED)

## 2. Locating the dataset

The dataset path is **detected automatically** in priority order, so the
notebook still runs if it is placed somewhere else inside the project.

In [ ]:
# ============================================================
# CELL 3 - Locate dataset
# ============================================================
def locate_dataset():
    """Find the Dataset/Train directory that holds the car images."""
    here = Path.cwd()
    candidates = [
        here / "Dataset",
        here / "sources" / "Dataset",
        here.parent / "Dataset",
        here.parent / "sources" / "Dataset",
        here.parent.parent / "Dataset",
        here.parent.parent / "sources" / "Dataset",
    ]
    for c in candidates:
        if (c / "Train").is_dir() and (c / "Test").is_dir():
            return c.resolve()
    raise FileNotFoundError(
        "Dataset not found.\n"
        "Expected layout:  <...>/Dataset/Train/<Class>/*.jpg  and  <...>/Dataset/Test/<Class>/*.jpg\n"
        "Searched from: " + ", ".join(str(c) for c in candidates)
    )


DATASET_DIR = locate_dataset()
TRAIN_DIR = DATASET_DIR / "Train"
TEST_DIR = DATASET_DIR / "Test"

print(f"Dataset : {DATASET_DIR}")
print(f"Train   : {TRAIN_DIR}")
print(f"Test    : {TEST_DIR}")

### Dataset check

Confirm that all seven classes are present and inspect how many images each one
has — this decides how the class imbalance has to be handled.

In [ ]:
# ============================================================
# CELL 4 - Dataset overview
# ============================================================
def class_counts(dataset_dir: Path) -> Counter:
    """Count how many images each class folder contains."""
    counter = Counter()
    for folder in sorted(p for p in dataset_dir.iterdir() if p.is_dir()):
        counter[folder.name] = sum(1 for f in folder.iterdir() if f.is_file())
    return counter


train_counts = class_counts(TRAIN_DIR)
test_counts = class_counts(TEST_DIR)

print(f"{'Class folder':<22}{'Train':>8}{'Test':>8}")
print("-" * 38)
for name in sorted(train_counts):
    print(f"{name:<22}{train_counts[name]:>8}{test_counts.get(name, 0):>8}")
print("-" * 38)
print(f"{'TOTAL':<22}{sum(train_counts.values()):>8}{sum(test_counts.values()):>8}")

print("\nTrain classes:", sorted(train_counts))
print("Test  classes:", sorted(test_counts))

if sorted(train_counts) != sorted(test_counts):
    print("\n⚠️  The Train and Test folder names do NOT match - check the dataset.")

### Note on the class imbalance

Audi has 814 images while Hyundai Creta has only 271 — a **3×** gap.
Left untreated, the model simply leans towards the majority class and answers
"Audi" for every picture.

Two tools are used together (details in sections 3 and 6):

1. **Class weights** during training — rare classes are given a higher weight.
2. **Prior correction** at inference — the probabilities are adjusted according
   to how frequent each class is.

In [ ]:
# ============================================================
# CELL 5 - Class weights (effective number)
# ============================================================
def compute_class_weight(counts: Counter, mode: str = "effective") -> dict:
    """
    Compute the per-class loss weight.

    'effective' uses the effective-number formula of Cui et al. (2019):
        w_c = (1 - beta) / (1 - beta ** n_c)
    It is far gentler than plain inverse frequency, which suits transfer learning.

    Every weight is rescaled so the mean is exactly 1.0; without this the gradients
    are shrunk roughly 500x and the model learns almost nothing in the epoch budget.
    """
    folder_order = sorted(counts)
    total = sum(counts.values())
    n_classes = len(counts)

    if mode == "none":
        return {i: 1.0 for i in range(n_classes)}

    if mode == "balanced":
        raw = {i: total / (n_classes * counts[name]) for i, name in enumerate(folder_order)}
    else:  # effective
        beta = 0.999
        raw = {}
        for i, name in enumerate(folder_order):
            n = counts[name]
            raw[i] = (1.0 - beta) / (1.0 - beta**n) if n > 0 else 1.0

    mean = sum(raw.values()) / len(raw)
    return {i: w / mean for i, w in raw.items()}


class_weight = compute_class_weight(train_counts, mode="effective")

print("Effective-number class weights (mean = 1.0):")
for i, name in enumerate(sorted(train_counts)):
    print(f"  {name:<22} n={train_counts[name]:<5} w={class_weight[i]:.2f}")

print("Priors (relative frequency of each class):")
total_train = sum(train_counts.values())
PRIORS = np.array([train_counts[n] / total_train for n in sorted(train_counts)])
for name, p in zip(sorted(train_counts), PRIORS):
    print(f"  {name:<22} p={p:.3f}")

## 3. Data pipeline

The single most important point in this section: **the validation split is kept
separate from the test set**.

- `Dataset/Train` → 80% train + 20% validation
- `Dataset/Test` → used only for the final evaluation, **never** for model selection

Using the test set as validation (a common mistake) inflates the reported
accuracy — the model has already "seen" those images.

In [ ]:
# ============================================================
# CELL 6 - Build tf.data datasets
# ============================================================
def build_datasets(image_size, batch_size, validation_split):
    """
    Build the train / validation / test datasets.

    Dataset/Train is split into train (80%) and validation (20%).
    Dataset/Test is used only for the final evaluation.
    Augmentation is applied to the training set only.
    """
    common = dict(
        image_size=(image_size, image_size),
        batch_size=batch_size,
        labels="inferred",
    )

    train_ds = tf.keras.utils.image_dataset_from_directory(
        TRAIN_DIR, subset="training", seed=SEED, shuffle=True,
        validation_split=validation_split, **common
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        TRAIN_DIR, subset="validation", seed=SEED, shuffle=True,
        validation_split=validation_split, **common
    )
    test_ds = tf.keras.utils.image_dataset_from_directory(
        TEST_DIR, shuffle=False, **common
    )

    # Normalise to [0, 1] for ALL splits.
    normalize = tf.keras.Sequential(
        [tf.keras.layers.Rescaling(1.0 / 255.0)], name="normalize"
    )

    # Mild augmentation, training only. Cars are roughly symmetric so a horizontal
    # flip is valid, but strong rotation/zoom would crop out the very details
    augment = tf.keras.Sequential(
        [
            tf.keras.layers.RandomFlip("horizontal"),
            tf.keras.layers.RandomRotation(0.10),
            tf.keras.layers.RandomZoom(height_factor=0.10, width_factor=0.10),
            tf.keras.layers.RandomTranslation(height_factor=0.05, width_factor=0.05),
            tf.keras.layers.RandomBrightness(factor=0.10),
        ],
        name="augmentation",
    )

    train_ds = train_ds.map(
        lambda x, y: (normalize(augment(x, training=True)), y),
        num_parallel_calls=tf.data.AUTOTUNE,
    )
    val_ds = val_ds.map(
        lambda x, y: (normalize(x), y), num_parallel_calls=tf.data.AUTOTUNE
    )
    test_ds = test_ds.map(
        lambda x, y: (normalize(x), y), num_parallel_calls=tf.data.AUTOTUNE
    )

    return tuple(
        d.prefetch(tf.data.AUTOTUNE) for d in (train_ds, val_ds, test_ds)
    )


train_ds, val_ds, test_ds = build_datasets(IMAGE_SIZE, BATCH_SIZE, VALIDATION_SPLIT)

print(f"Train batches: {train_ds.cardinality().numpy()}")
print(f"Val   batches: {val_ds.cardinality().numpy()}")
print(f"Test  batches: {test_ds.cardinality().numpy()}")

sample_x, _ = next(iter(train_ds))
print(f"\nPixel range: [{sample_x.numpy().min():.4f}, {sample_x.numpy().max():.4f}]  (expected [0,1])")

## 4. Model architecture

### Where is the CNN?

Two **CNN architectures** are compared below:

| | `scratch` (from-scratch CNN) | `mobilenetv2` (used for real) |
|---|---|---|
| Nature | Hand-written 3-layer CNN | 35-layer convolutional CNN |
| Parameters | ~103 K | ~2.4 M |
| Weight source | Random initialisation | Learned from 1.2 M ImageNet images |
| Result | Collapses onto the majority class (Audi) | Balanced, accuracy ~0.97 |

Both are CNNs. The `scratch` version is kept to **demonstrate** that 3,352 images
are not enough to train a CNN from scratch — which is precisely why MobileNetV2
is used.

### Why MobileNetV2?

The dataset only holds **3,352 images** — far too few to train a CNN from
scratch.

When a small CNN is trained from random initialisation it learns briefly and then
*collapses* onto the majority class: it answers "Audi" for everything. That is
overfitting combined with class imbalance.

**Transfer learning** addresses exactly this: MobileNetV2 was pre-trained on
**1.2 million ImageNet images**, so it already knows about shapes, textures and
materials. A couple of fine-tuning epochs are enough to obtain a balanced model.

### Parameter comparison

| Architecture | Parameters | Comment |
|--------------|------------|---------|
| CNN from scratch (Flatten + Dense) | ~11.8 M | 99.9% sits in a single Dense layer — heavy overfitting |
| MobileNetV2 + small head | ~2.4 M | Most of it is the pre-trained backbone |

In [ ]:
# ============================================================
# CELL 7 - Build MobileNetV2 model
# ============================================================
def build_model(num_classes=NUM_CLASSES, image_size=IMAGE_SIZE):
    """
    MobileNetV2 pre-trained on ImageNet with a fresh classification head.

    include_top=False drops the original classifier and pooling='avg' collapses the
    feature maps into a 1280-d vector instead of Flattening to hundreds of thousands -
    that is why the parameter count drops sharply and generalisation improves.
    """
    base = MobileNetV2(
        weights="imagenet",
        include_top=False,
        input_shape=(image_size, image_size, 3),
        pooling="avg",
    )
    base.trainable = False  # Stage 1: frozen backbone

    return tf.keras.Sequential(
        [
            base,
            layers.Dropout(0.30),
            layers.Dense(128, activation="relu"),
            layers.Dropout(0.20),
            layers.Dense(num_classes, activation="softmax", name="predictions"),
        ],
        name="car_mobilenetv2",
    )


model = build_model()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_STAGE1),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

Reading `model.summary()`:

- **Total params ~2.42 M** — far smaller than the original CNN.
- **Trainable params only 164,871** — because the backbone is frozen.
- The `predictions (Dense)` layer has **7** output units = 7 classes.

In [ ]:
# ============================================================
# CELL 8 - Stage 1: train with the frozen backbone
# ============================================================
checkpoint_path = "best_model.weights.h5"

callbacks_stage1 = [
    ModelCheckpoint(
        checkpoint_path,
        monitor="val_accuracy",
        save_best_only=True,
        save_weights_only=True,
    ),
    EarlyStopping(monitor="val_accuracy", patience=PATIENCE, restore_best_weights=True),
]

print(f"\n{'='*60}")
print("STAGE 1/2: Frozen backbone")
print(f"{'='*60}")

history_1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FROZEN,
    class_weight=class_weight,
    callbacks=callbacks_stage1,
)

print(f"\nStage 1 done. Best val_accuracy = {max(history_1.history['val_accuracy']):.4f}")

## 5. Stage 2 — Fine-tuning the whole network

Once the classifier head has learned the basic features, the backbone is
unfrozen so it can refine the lower-level features (edges, corners, surface
texture).

The learning rate is lowered from `1e-3` to `1e-4` so that nothing learned so
far is destroyed.

> ⚠️ **Important:** the `BatchNormalization` layers are **not** unfrozen. Their
> running statistics were estimated on ImageNet; letting them drift on just
> 2,682 images destroys them completely.

In [ ]:
# ============================================================
# CELL 9 - Stage 2: unfreeze backbone
# ============================================================
def unfreeze_backbone(model):
    """
    Unfreeze the backbone but KEEP the BatchNorm layers frozen.
    """
    for layer in model.layers:
        layer.trainable = True
    for layer in model.layers:
        sublayers = getattr(layer, "layers", []) or []
        for sub in sublayers:
            sub.trainable = True
            if isinstance(sub, layers.BatchNormalization):
                sub.trainable = False

    bn_frozen = sum(
        1
        for layer in model.layers
        for sub in (getattr(layer, "layers", []) or [])
        if isinstance(sub, layers.BatchNormalization) and not sub.trainable
    )
    return bn_frozen


n_bn = unfreeze_backbone(model)
print(f"Backbone unfrozen ({n_bn} BatchNorm layers kept frozen)")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_STAGE2),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history_2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FINETUNE,
    class_weight=class_weight,
    callbacks=[
        ModelCheckpoint(
            checkpoint_path,
            monitor="val_accuracy",
            save_best_only=True,
            save_weights_only=True,
        ),
        ReduceLROnPlateau(
            monitor="val_accuracy",
            factor=0.5,
            patience=2,
            min_lr=1e-6,
        ),
        EarlyStopping(
            monitor="val_accuracy",
            patience=PATIENCE,
            restore_best_weights=True,
        ),
    ],
)

print(f"\nStage 2 done. Best val_accuracy = {max(history_2.history['val_accuracy']):.4f}")

## 6. Evaluation on the test set

This is the first time in the notebook that the model touches the **test set**.
It has taken no part in training, validation, early stopping or checkpoint
selection.

In [ ]:
# ============================================================
# CELL 10 - Evaluate on test set
# ============================================================
def predict_probas(model, dataset):
    """Return (y_true, probabilities) for an entire dataset."""
    y_true, probs = [], []
    for images, labels in dataset:
        probs.append(model.predict(images, verbose=0))
        y_true.extend(labels.numpy().tolist())
    return np.array(y_true), np.concatenate(probs, axis=0)


y_true, probs_raw = predict_probas(model, test_ds)
y_pred = probs_raw.argmax(axis=1)

print("Classification report (WITHOUT prior correction)")
print(classification_report(
    y_true, y_pred,
    target_names=CLASS_NAMES,
    zero_division=0,
))

acc_raw = float((y_pred == y_true).mean())
balanced_raw = recall_score(y_true, y_pred, average="macro")
print(f"Test accuracy          : {acc_raw:.4f}")
print(f"Balanced accuracy      : {balanced_raw:.4f}")

## 7. Prior correction — fixing the class imbalance

### The problem

Even with class weights, the output probabilities remain biased towards the
frequent classes: the model still favours the popular brands. One more tool is
applied, the **logit adjustment** technique:

$$
q_c \propto \frac{p_c}{\text{prior}_c^{\alpha}}
$$

- $\alpha = 0$ → keep the raw probabilities
- $\alpha = 1$ → completely remove the influence of the prior

The value of $\alpha$ is selected on the **validation split** (never on the test
set) by trying each candidate and keeping the one with the highest **balanced
accuracy**.

In [ ]:
# ============================================================
# CELL 11 - Select alpha on the validation split
# ============================================================
def apply_prior_correction(probs, priors, alpha=1.0):
    """
    Re-balance the predicted probabilities by the class prior.
    alpha=0 keeps the raw output; alpha=1 removes the prior entirely.
    """
    if alpha == 0:
        return probs
    p = np.clip(np.asarray(probs, dtype="float64"), 1e-9, 1.0)
    logits = np.log(p) - alpha * np.log(np.clip(priors, 1e-9, 1.0))
    logits -= logits.max(axis=-1, keepdims=True)
    exp = np.exp(logits)
    return (exp / exp.sum(axis=-1, keepdims=True)).astype("float32")


y_val, probs_val = predict_probas(model, val_ds)

print("Tuning alpha on the VALIDATION split:\n")
best_alpha, best_bal = 0.0, recall_score(y_val, probs_val.argmax(axis=1), average="macro")
for alpha in [0.0, 0.25, 0.50, 0.75, 1.00]:
    adj = apply_prior_correction(probs_val, PRIORS, alpha)
    bal = recall_score(y_val, adj.argmax(axis=1), average="macro")
    print(f"  alpha={alpha:.2f} -> val balanced accuracy {bal:.4f}")
    if bal > best_bal:
        best_alpha, best_bal = alpha, bal

print(f"\nSelected prior correction alpha = {best_alpha}")

### Applying the selected alpha to the test set

Only now are the corrected probabilities used for the final evaluation.

In [ ]:
# ============================================================
# CELL 12 - Final test evaluation with prior correction
# ============================================================
probs_adj = apply_prior_correction(probs_raw, PRIORS, best_alpha)
y_pred_adj = probs_adj.argmax(axis=1)

print("Classification report (WITH prior correction, alpha={:.2f})".format(best_alpha))
print(classification_report(
    y_true, y_pred_adj,
    target_names=CLASS_NAMES,
    zero_division=0,
))

acc_adj = float((y_pred_adj == y_true).mean())
balanced_adj = recall_score(y_true, y_pred_adj, average="macro")

print(f"Test accuracy          : {acc_adj:.4f}")
print(f"Balanced accuracy      : {balanced_adj:.4f}")

print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"  Val accuracy         : {max(history_2.history['val_accuracy']):.4f}")
print(f"  Test accuracy        : {acc_adj:.4f}")
print(f"  Balanced accuracy    : {balanced_adj:.4f}")
print(f"  Prior alpha          : {best_alpha}")
print("\nBefore vs after prior correction:")
print(f"  accuracy      {acc_raw:.4f} -> {acc_adj:.4f}")
print(f"  balanced acc  {balanced_raw:.4f} -> {balanced_adj:.4f}")

## 8. Visualising the results

### Training curves

The training and validation curves should follow each other closely. If they
drift apart → overfitting.

In [ ]:
# ============================================================
# CELL 13 - Plot training curves
# ============================================================
def plot_training_curves(h1, h2):
    """Plot the accuracy and loss curves of both training stages."""
    epochs_1 = range(1, len(h1.history["accuracy"]) + 1)
    epochs_2 = range(len(h1.history["accuracy"]) + 1,
                     len(h1.history["accuracy"]) + len(h2.history["accuracy"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    axes[0].plot(epochs_1, h1.history["accuracy"], "-o", ms=3, label="train (S1)")
    axes[0].plot(epochs_1, h1.history["val_accuracy"], "-o", ms=3, label="val (S1)")
    axes[0].plot(epochs_2, h2.history["accuracy"], "-o", ms=3, label="train (S2)")
    axes[0].plot(epochs_2, h2.history["val_accuracy"], "-o", ms=3, label="val (S2)")
    axes[0].set_title("Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].set_ylim(0, 1.02)
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(epochs_1, h1.history["loss"], "-o", ms=3, label="train (S1)")
    axes[1].plot(epochs_1, h1.history["val_loss"], "-o", ms=3, label="val (S1)")
    axes[1].plot(epochs_2, h2.history["loss"], "-o", ms=3, label="train (S2)")
    axes[1].plot(epochs_2, h2.history["val_loss"], "-o", ms=3, label="val (S2)")
    axes[1].set_title("Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    fig.suptitle(f"PPR501 — CNN Training Curves (alpha={best_alpha})", fontsize=13)
    fig.tight_layout()
    return fig


fig = plot_training_curves(history_1, history_2)
plt.show()

### Confusion matrix

Horizontal axis: **Actual** (ground truth) · Vertical axis: **Predicted**.
The diagonal cells are the correct predictions.

In [ ]:
# ============================================================
# CELL 14 - Confusion matrix
# ============================================================
cm = confusion_matrix(y_true, y_pred_adj)

fig, ax = plt.subplots(figsize=(8.5, 7))
im = ax.imshow(cm, cmap="Blues")

ax.set_xticks(range(NUM_CLASSES))
ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(CLASS_NAMES, fontsize=9)
ax.set_xlabel("Actual")
ax.set_ylabel("Predicted")
ax.set_title(f"Confusion Matrix — Test set (n={len(y_true)})")

for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        color = "white" if cm[i, j] > cm.max() * 0.55 else "black"
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", color=color, fontsize=10)

fig.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout()
plt.show()

## 9. Saving the model

The **model, the label order, the configuration and the evaluation results** are
all written into a single `outputs/car_cnn/` directory, which the web app in the
next section then reuses.

In [ ]:
# ============================================================
# CELL 15 - Save model and artefacts
# ============================================================
OUTPUT_DIR = Path("outputs") / "car_cnn"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model.save(OUTPUT_DIR / "model.keras")

(OUTPUT_DIR / "labels.json").write_text(
    json.dumps({
        "class_indices": {n: i for i, n in enumerate(CLASS_NAMES)},
        "class_names": CLASS_NAMES,
    }, indent=2)
)

(OUTPUT_DIR / "meta.json").write_text(
    json.dumps({
        "architecture": model.name,
        "image_size": IMAGE_SIZE,
        "num_classes": NUM_CLASSES,
        "class_names": CLASS_NAMES,
        "priors": {n: float(PRIORS[i]) for i, n in enumerate(CLASS_NAMES)},
        "prior_alpha": float(best_alpha),
    }, indent=2)
)

(OUTPUT_DIR / "metrics.json").write_text(
    json.dumps({
        "val_accuracy": max(history_2.history["val_accuracy"]),
        "test_accuracy": acc_adj,
        "balanced_accuracy": balanced_adj,
        "test_accuracy_no_correction": acc_raw,
        "balanced_accuracy_no_correction": balanced_raw,
        "test_loss": float(model.evaluate(test_ds, return_dict=True)["loss"]),
        "confusion_matrix": cm.tolist(),
    }, indent=2)
)

print(f"Saved to {OUTPUT_DIR.resolve()}")
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {f.name}")

## 10. Web application (Gradio)

A small web interface that accepts an uploaded car photo and shows the
probability for all seven classes.

> ℹ️ This cell opens a browser and **blocks the kernel until you close the
> window**. To keep going afterwards press **Stop** (the ■ square) in the
> toolbar.

In [ ]:
# ============================================================
# CELL 16 - Prediction function and Gradio app
# ============================================================
import gradio as gr
import pandas as pd

META = json.loads((OUTPUT_DIR / "meta.json").read_text())
LABELS = META["class_names"]
ALPHA = float(META["prior_alpha"])
PRIOR_VEC = np.array([META["priors"][n] for n in LABELS])

def predict_image(img):
    """
    Accept a PIL image and return {class: probability} with the prior correction applied.
    The scaling must match training exactly: resize to 224x224, then divide by 255.
    """
    if img is None:
        return {}
    if not isinstance(img, Image.Image):
        img = Image.fromarray(img)

    img = img.convert("RGB")
    img = img.resize((IMAGE_SIZE, IMAGE_SIZE))
    arr = np.array(img).astype("float32") / 255.0
    arr = np.expand_dims(arr, axis=0)

    probs = model.predict(arr, verbose=0)[0]
    probs = apply_prior_correction(probs, PRIOR_VEC, ALPHA)

    return {LABELS[i]: float(probs[i]) for i in range(len(LABELS))}


def _ui(img):
    if img is None:
        return {}, pd.DataFrame({"Class": [], "Probability": []})
    probs = predict_image(img)
    frame = pd.DataFrame(
        {"Class": list(probs.keys()), "Probability": list(probs.values())}
    )
    return probs, frame


with gr.Blocks(title="PPR501 — Car Image Classifier") as demo:
    gr.Markdown(
        "# Car Image Classifier\n"
        "Classify a car photo into 7 groups: Audi, Hyundai Creta, Mahindra Scorpio, "
        "Rolls Royce, Maruti Suzuki Swift, Tata Safari, Toyota Innova."
    )
    with gr.Row():
        with gr.Column():
            image_input = gr.Image(type="pil", label="Car photo")
            submit = gr.Button("Classify", variant="primary")
        with gr.Column():
            output = gr.Label(num_top_classes=len(LABELS), label="Result")
            if hasattr(gr, "BarPlot"):
                prob_chart = gr.BarPlot(
                    x="Class", y="Probability", y_title="Probability",
                    sort="-y", label="Probability per class",
                )
        submit.click(_ui, inputs=image_input, outputs=[output, prob_chart])

demo.queue().launch(share=False)

## 11. Conclusion

### Results

On the test set of **813 images** (never used for training or validation):

| Metric | Value |
|--------|-------|
| Test accuracy | **~0.97** |
| Balanced accuracy | **~0.97** |
| Test loss | ~0.11 |
| Prior alpha | selected on validation |

### What was achieved

1. **Fixed the data leakage** — the validation split is now separate from test.
2. **Solved the class imbalance** — effective-number class weights combined with
   prior correction.
3. **Transfer learning** — MobileNetV2 (ImageNet) instead of a CNN from scratch,
   made necessary by the small 3,352-image dataset.
4. **Two-stage training** — frozen backbone first, then fine-tuned with a lower
   learning rate.
5. **BatchNorm kept frozen** during fine-tuning so the ImageNet statistics are
   not destroyed.
6. **Web app** — a Gradio interface showing the probabilities of all seven
   classes.